# Explainability Analysis
This notebook calculates and visualizes global/local interpretations using SHAP, LIME, and Grad-CAM.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import load_or_create_datasets
from src.data.preprocessing import preprocess_ecg_data, preprocess_tabular_data
from src.explainability.visualize_explanations import (
    plot_ecg_gradcam,
    plot_lime_explanation,
    plot_shap_summary,
)
from src.models.cnn_ecg import ECGCNN
from src.models.gradcam_explainer import GradCAMExplainer
from src.models.lime_explainer import LIMEExplainerWrapper
from src.models.xgboost_model import XGBoostModelWrapper
from src.utils.config import Config

config = Config(str(PROJECT_ROOT / "configs/params.yaml"), str(PROJECT_ROOT / "configs/paths.yaml"))
output_dir = Path(config.get_path("outputs")["outputs_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
df, ecg = load_or_create_datasets(config)
X_train, X_test, y_train, y_test, _ = preprocess_tabular_data(df, config)
processed_ecg = preprocess_ecg_data(ecg, config)

xgb_wrapper = XGBoostModelWrapper(config)
xgb_wrapper.load(config.get_path("models")["xgboost_path"])
shap_values = xgb_wrapper.get_shap_values(X_test)
plot_shap_summary(shap_values, X_test, config.get_path("outputs")["shap_plot_path"])
print(f"SHAP explanation generated for {len(X_test)} test patients.")

In [ ]:
lime_explainer = LIMEExplainerWrapper(
    training_data=X_train.to_numpy(),
    feature_names=X_train.columns.tolist(),
    class_names=["No CVD", "CVD"],
)
lime_features = lime_explainer.explain_instance(
    X_test.iloc[0].to_numpy(),
    xgb_wrapper.predict_proba,
    num_features=min(8, X_test.shape[1]),
)
print("Top local LIME contributions:")
print(lime_features)
plot_lime_explanation(
    lime_features,
    str(output_dir / "lime_local_explanation.png"),
)

In [ ]:
cnn_model = ECGCNN(in_channels=processed_ecg.shape[1], num_classes=2)
cnn_model.load_state_dict(torch.load(config.get_path("models")["ecg_cnn_path"], map_location="cpu"))
gradcam = GradCAMExplainer(cnn_model)
test_signal = processed_ecg[X_test.index[0]]
input_tensor = torch.tensor(test_signal[None, ...], dtype=torch.float32)
heatmap = gradcam.generate_heatmap(input_tensor, target_class=1)
plot_ecg_gradcam(
    test_signal,
    heatmap,
    config.get_path("outputs")["gradcam_plot_path"],
    lead_idx=0,
)
print("Grad-CAM explanation generated for the first test ECG.")